# BME 574 — Worked example (OPTIONAL)
## The full pipeline, run end to end on the Yale faces data

> ## This notebook is optional and is not part of Project 1.
>
> **Project 1 is defined by `BME574_Project1_starter.ipynb` and the project handout.** Nothing
> here is required, nothing here is graded, and you can complete the project without opening it.
>
> It goes deliberately *beyond* what the project asks for — a second prediction target, three
> different splits compared, two forms of leakage measured, and a component-ranking step. Read it
> if you have finished the project and want to see how much further the same machinery goes, or
> if you are curious how the face lecture connects to the biomedical tracks.

**If you are starting the project, go to the starter notebook instead.**

---

This notebook walks the full pipeline on the face data from lecture, so you can see a complete,
valid analysis where you already know what the answers should look like.

**Faces are not a project option.** They are here because they make the machinery visible —
including the two ways it silently goes wrong. Notes marked `→ In your track` map each step onto
the biomedical tracks.

---

### What this notebook demonstrates that the class notebook did not

| | Class notebook | This notebook |
|---|---|---|
| Basis fitted on | All of persons 1–36 | **Training rows only** |
| Evaluation | Visual inspection of reconstructions | **Held-out accuracy vs. three baselines** |
| Choice of *k* | Fixed list, chosen by eye | **Swept, with the operating point justified** |
| Splitting | None | **Three splits compared, two leaks quantified** |
| Components plotted | Modes 5 and 6, chosen for these two people | **Ranked by discriminability, criterion stated** |
| Second target | — | **In-gallery vs. impostor detection** |

### The two leaks this notebook measures

1. **Basis leakage** — fitting the SVD on all images, test rows included. §2.4 of the handout.
2. **Near-duplicate leakage** — consecutive images of one subject are adjacent illumination
   conditions and are nearly identical. A random split puts near-duplicates on both sides.

Both are invisible unless you look for them, and both have exact analogues in the biomedical
tracks — beats from one patient (T2), epochs from one night (T4), cycles from one walker (T5).

**They are not equally large here, and the notebook measures rather than assumes.** On this data
the near-duplicate leak is worth tens of accuracy points and the basis leak is worth almost
nothing — because the test set is a small fraction of a matrix with far more columns than rows,
so adding it barely moves the basis. Do not generalize that to your track: with a small dataset,
or a test set that is a third of your rows, the basis leak can dominate instead. The lesson is to
run the comparison, not to memorize which leak is worse.

---
## 0 · Setup and data

The loader expects `allFaces.mat` from the course data archive — the same file the class notebook
used, in the same relative location. **If it is not found, the notebook builds a synthetic
surrogate with identical structure** (38 subjects, uneven image counts, 192×168 images, dominant
illumination variation) so that every cell still runs. The surrogate exists to exercise the code,
not to teach you about faces — a banner tells you which one you are looking at.

In [ ]:
import os, json
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score, balanced_accuracy_score

SEED = 0
rng = np.random.default_rng(SEED)

plt.rcParams['figure.figsize'] = [7, 5]
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})

In [ ]:
DATA_PATH = os.path.join('..', 'DATA', 'allFaces.mat')


def _synthetic(n_subj=38, m=168, n=192, seed=0):
    """Yale-B-shaped surrogate: same conventions, same failure modes.

    faces is (m*n, total) COLUMN-major, nfaces is per-subject counts, counts are uneven,
    and illumination dominates identity — exactly as in the real data.
    """
    g = np.random.default_rng(seed)
    nfaces = g.integers(50, 65, size=n_subj)
    yy, xx = np.mgrid[0:n, 0:m] / max(m, n)
    faces = np.zeros((m * n, int(nfaces.sum())), dtype=float)
    col = 0
    for s in range(n_subj):
        # a per-subject 'identity' image, smooth and low-rank
        base = np.zeros((n, m))
        for _ in range(6):
            cy, cx = g.uniform(0.15, 0.85, 2)
            w = g.uniform(0.05, 0.22)
            base += g.normal() * np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * w ** 2))
        base = (base - base.min()) / (np.ptp(base) + 1e-12)
        for j in range(int(nfaces[s])):
            # illumination: a directional ramp, the dominant mode in real Yale B too
            az = np.cos(2 * np.pi * j / nfaces[s])
            el = np.sin(2 * np.pi * j / nfaces[s])
            light = 0.5 + 0.75 * (az * (xx - 0.5) + el * (yy - 0.5))
            img = base * np.clip(light, 0.05, None) + 0.02 * g.normal(size=(n, m))
            faces[:, col] = img.T.reshape(-1)   # .T so reshape(m,n).T recovers the image
            col += 1
    return faces, m, n, nfaces


if os.path.exists(DATA_PATH):
    mat = scipy.io.loadmat(DATA_PATH)
    faces = mat['faces'].astype(float)
    m = int(mat['m'][0][0])
    n = int(mat['n'][0][0])
    nfaces = np.ndarray.flatten(mat['nfaces'])
    REAL_DATA = True
else:
    faces, m, n, nfaces = _synthetic(seed=SEED)
    REAL_DATA = False

# ---- downsample -------------------------------------------------------
# Full resolution is 192x168 = 32,256 columns. At float64 the full matrix is
# ~600 MB and the SVD is slow on a laptop. DS=2 keeps every conclusion in this
# notebook intact at a quarter of the cost. Set DS=1 to work at full resolution.
DS = 2
if DS > 1:
    imgs = faces.T.reshape(-1, m, n)[:, ::DS, ::DS]
    m, n = imgs.shape[1], imgs.shape[2]
    faces = imgs.reshape(len(imgs), -1).T

banner = 'REAL Yale B data' if REAL_DATA else 'SYNTHETIC SURROGATE (allFaces.mat not found)'
print('=' * 62)
print(f'  {banner}')
print('=' * 62)
print(f'faces   : {faces.shape}   (pixels x images, COLUMN-major)')
print(f'image   : {n} x {m} after reshape(m, n).T   (downsample DS={DS})')
print(f'subjects: {len(nfaces)}, images per subject {nfaces.min()}-{nfaces.max()},'
      f' total {int(nfaces.sum())}')

### Orientation — the transpose that bites everyone

`allFaces.mat` stores images as **columns** (Brunton & Kutz convention). The specification, and
scikit-learn, want observations as **rows**. Transpose once, here, and never think about it again.

> **→ In your track.** T1 gives you `(N, 28, 28)`, so `.reshape(N, -1)` and you are already
> row-major. T2 and T3 give you time × channels, and you must decide which is the observation.
> Print the shape after every reshape.

In [ ]:
X_all = faces.T                      # (n_images, n_pixels) — rows are observations
n_images, n_pixels = X_all.shape

# subject id for every row, from the cumulative image counts
starts = np.concatenate([[0], np.cumsum(nfaces)]).astype(int)
subject = np.zeros(n_images, dtype=int)
for s in range(len(nfaces)):
    subject[starts[s]:starts[s + 1]] = s

# index of each image WITHIN its subject — this is the illumination condition
within = np.concatenate([np.arange(int(c)) for c in nfaces])

print('X_all   :', X_all.shape)
print('subject :', subject.shape, '| unique', len(np.unique(subject)))
print('within  :', within.shape, '| max', within.max())
assert len(subject) == len(within) == n_images
assert n_pixels == m * n

---
## S1 · Frame the question

Two targets, because faces support two genuinely different problems and the biomedical tracks
each look like one of them.

**Target A — identity (multi-class).** Given a new photograph of someone in the gallery, which
of the 36 gallery subjects is it? This is what we did in class.

**Target B — in-gallery vs. impostor (binary).** Given a photograph, is this person in the
gallery at all? Answered by the *reconstruction residual*: a face the eigenbasis has never seen
reconstructs badly. This is Turk & Pentland's "distance from face space", and it is the same
computation as anomaly detection and automated quality control in medical imaging.

**Unit of observation.** One image. Not one subject — a subject contributes many rows.

> **→ In your track.** Target A is T1, T3, T4 and T6: predict a class for each observation.
> Target B is the shape of quality-control and out-of-distribution problems. Whichever you
> choose, name it in one sentence before you write any code, and say what decision it stands in
> for.

### A note on subject-level splitting

The handout insists you split by subject. **Target A is the exception that proves the rule:**
identity *is* the target, so the same person must appear in training and test — otherwise the
question is unanswerable. That is legitimate here and illegitimate in T2, where the target is
arrhythmia and the patient is a nuisance variable. Ask yourself: *is the subject the thing I am
predicting, or the thing I am predicting through?* If it is the latter, split by subject.

In [ ]:
GALLERY = np.arange(36)          # subjects 0-35: the gallery (as in class)
IMPOSTOR = np.arange(36, len(nfaces))   # subjects 36+: never enrolled

gal_mask = np.isin(subject, GALLERY)
imp_mask = np.isin(subject, IMPOSTOR)

print(f'gallery images : {gal_mask.sum():5d}  ({len(GALLERY)} subjects)')
print(f'impostor images: {imp_mask.sum():5d}  ({len(IMPOSTOR)} subjects)')

---
## S2 · Split before you look

Before any mean, any scaler, any SVD. We build **three** splits of the gallery so we can measure
what a careless one costs:

| Split | How | What it is for |
|---|---|---|
| `random` | Random 70/30 over images | The tempting, wrong one |
| `block` | First 70% of each subject's images train, last 30% test | Honest: separates illumination conditions |
| `impostor` | Entire subjects held out | Target B — people never enrolled |

The `random` split puts illumination condition *j* and condition *j+1* of the same subject on
opposite sides. Those two images are near-duplicates. The model does not have to generalize; it
only has to remember.

> **→ In your track.** `random` is what you get from `train_test_split` with no `groups=`
> argument, and it is what will silently inflate your T2, T4, T5 and T6 results. Use
> `GroupShuffleSplit` or `GroupKFold` with the subject id as the group. T3 hands you
> `strat_fold`; use it.

In [ ]:
gal_idx = np.where(gal_mask)[0]

# --- random split (the tempting, wrong one) ---
perm = rng.permutation(gal_idx)
cut = int(0.7 * len(perm))
split_random = {'train': np.sort(perm[:cut]), 'test': np.sort(perm[cut:])}

# --- block split (honest: hold out the LAST illumination conditions of each subject) ---
tr_b, te_b = [], []
for s in GALLERY:
    idx = np.where(subject == s)[0]          # already ordered by condition
    k = int(0.7 * len(idx))
    tr_b.append(idx[:k]); te_b.append(idx[k:])
split_block = {'train': np.concatenate(tr_b), 'test': np.concatenate(te_b)}

SPLITS = {'random': split_random, 'block': split_block}

for name, sp in SPLITS.items():
    assert len(np.intersect1d(sp['train'], sp['test'])) == 0, 'train/test overlap!'
    print(f"{name:7s}  train {len(sp['train']):5d}  test {len(sp['test']):5d}")

# S2 requires the split be reproducible from disk
np.savez('splits.npz', **{f'{k}_{p}': v for k, sp in SPLITS.items() for p, v in sp.items()})
print('\nsplit indices written to splits.npz')

---
## S3 · Build and justify the matrix

Three sentences, as the specification demands:

1. **Rows are images, columns are pixels**, because the question is asked of a whole photograph
   and the modes we want live in pixel space.
2. **We centre** on the training mean, because we want modes of variation about an average face
   (PCA), not a low-rank approximation of the raw pixel values.
3. **We do not standardize per pixel**, because every column is the same physical quantity in the
   same units — an 8-bit intensity. Scaling would amplify pixels that happen to sit in a corner
   where nothing ever happens.

> **→ In your track.** Sentence 3 flips for T5 and T6, where columns are different sensors with
> different sensitivities and standardizing is mandatory. It flips for T4 too, once you decide
> between spectral shape and absolute power.

### Alignment

The cropped Yale images are pre-registered — eyes in the same pixels, faces at the same scale.
That is why this works at all, and it is invisible until you break it. The cell below breaks it
deliberately.

In [ ]:
def fit_basis(X, k=None):
    """Return (mu, Vt) fitted on X. This is the model. It must see training rows only."""
    mu = X.mean(axis=0)
    U, s, Vt = np.linalg.svd(X - mu, full_matrices=False)
    return mu, s, (Vt if k is None else Vt[:k])


def project(X, mu, Vt, k):
    return (X - mu) @ Vt[:k].T


tr = SPLITS['block']['train']
mu_tr, s_tr, Vt_tr = fit_basis(X_all[tr])
print('basis fitted on', len(tr), 'training rows')
print('Vt:', Vt_tr.shape, '| singular values:', s_tr.shape)

#### What misalignment costs — a two-line experiment

Roll each image by a random shift and refit. Watch the singular value spectrum flatten: the
energy that was concentrated in a few identity modes is now spread across many translation modes.

In [ ]:
shifted = X_all[tr].reshape(-1, m, n).copy()
for i in range(len(shifted)):
    shifted[i] = np.roll(shifted[i], rng.integers(-12, 13), axis=0)
_, s_shift, _ = fit_basis(shifted.reshape(len(shifted), -1))

e_aligned = np.cumsum(s_tr ** 2) / np.sum(s_tr ** 2)
e_shift = np.cumsum(s_shift ** 2) / np.sum(s_shift ** 2)
k90 = np.searchsorted(e_aligned, 0.90) + 1
k90s = np.searchsorted(e_shift, 0.90) + 1
print(f'components for 90% energy — aligned: {k90:4d}   misaligned: {k90s:4d}')
print(f'misalignment cost: {k90s / max(k90, 1):.1f}x more components for the same energy')

---
## S4 · Decompose the training set  ·  **F1** and **F2**

Two figures come out of this stage: the singular value spectrum with a defended choice of `k`,
and the Eckart–Young check that Assignment 1 requires.

In [ ]:
# ---------------- F1: singular value spectrum ----------------
energy = np.cumsum(s_tr ** 2) / np.sum(s_tr ** 2)
k_energy = int(np.searchsorted(energy, 0.95) + 1)          # 95% energy criterion

# elbow criterion: point of maximum distance to the chord of the log spectrum
ls = np.log10(s_tr[:400] + 1e-30)
xx = np.arange(len(ls))
chord = ls[0] + (ls[-1] - ls[0]) * xx / (len(ls) - 1)
k_elbow = int(np.argmax(chord - ls) + 1)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(s_tr, lw=1.2)
ax[0].set(xlabel='index', ylabel='singular value', title='F1a  spectrum (linear)')
ax[1].semilogy(s_tr, lw=1.2)
for kk, c, lab in [(k_energy, 'crimson', f'95% energy  k={k_energy}'),
                   (k_elbow, 'seagreen', f'elbow  k={k_elbow}')]:
    ax[1].axvline(kk, color=c, ls='--', lw=1.4, label=lab)
ax[1].set(xlabel='index', ylabel='singular value', title='F1b  spectrum (log)')
ax[1].legend(fontsize=9)
fig.tight_layout(); plt.show()

print(f'95% energy  -> k = {k_energy}')
print(f'elbow       -> k = {k_elbow}')

In [ ]:
# ---------------- F2: measured truncation error vs the Eckart-Young bound ----------------
A = X_all[tr] - mu_tr
U_a, s_a, Vt_a = np.linalg.svd(A, full_matrices=False)

ks = np.unique(np.clip(np.round(np.logspace(0, np.log10(min(400, len(s_a) - 1)), 22)).astype(int),
                       1, len(s_a) - 1))
measured = np.array([np.linalg.norm(A - (U_a[:, :k] * s_a[:k]) @ Vt_a[:k], 'fro') for k in ks])
bound = np.array([np.sqrt((s_a[k:] ** 2).sum()) for k in ks])

rel = np.abs(measured - bound) / np.maximum(bound, 1e-30)
print(f'max relative deviation from the bound: {rel.max():.3e}')
assert rel.max() < 1e-8, 'Eckart-Young check FAILED'
print('Eckart-Young check PASSED')

plt.figure(figsize=(6, 4.2))
plt.loglog(ks, measured, 'o', ms=7, mfc='none', mew=1.6, label='measured  $\\|A-A_k\\|_F$')
plt.loglog(ks, bound, '-', lw=1.4, label=r'bound  $\sqrt{\sum_{i>k}\sigma_i^2}$')
plt.xlabel('rank $k$'); plt.ylabel('Frobenius error')
plt.title('F2  truncation error matches the Eckart–Young bound')
plt.legend(); plt.tight_layout(); plt.show()

---
## **F3** · The leading modes, in their natural domain

For images the natural domain is an image. For T2 it is a beat waveform; for T4 a spectrum
plotted against frequency; for T6 a bar per muscle channel. **Caption every mode with what you
think it is** — and if the honest answer is "lighting", say lighting.

In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(14, 2.6))
axes[0].imshow(mu_tr.reshape(m, n).T, cmap='gray')
axes[0].set_title('mean', fontsize=10); axes[0].axis('off')
for i in range(6):
    axes[i + 1].imshow(Vt_tr[i].reshape(m, n).T, cmap='gray')
    axes[i + 1].set_title(f'mode {i + 1}', fontsize=10); axes[i + 1].axis('off')
fig.suptitle('F3  mean face and the first six modes', y=1.06)
fig.tight_layout(); plt.show()

print('Read these honestly. In the real Yale B data the leading modes are dominated by')
print('ILLUMINATION DIRECTION, not identity — lighting varies far more than people do.')
print('The same trap waits in T1, where brightness and anatomy outrank pathology.')

---
## S5 · Project — and the leak that lives here

This is §2.4 of the handout, made numerical. Below, the *only* difference between the two
pipelines is which rows the basis was fitted on. Everything else is identical.

In [ ]:
def nearest_centroid_fit(Z, y):
    classes = np.unique(y)
    return classes, np.stack([Z[y == c].mean(axis=0) for c in classes])


def nearest_centroid_predict(Z, classes, cents):
    d = ((Z[:, None, :] - cents[None, :, :]) ** 2).sum(-1)
    return classes[np.argmin(d, axis=1)]


def run(split, k, leak_basis=False):
    """Nearest-centroid identity accuracy. leak_basis=True fits the SVD on train+test."""
    tr, te = split['train'], split['test']
    if leak_basis:
        fit_rows = np.concatenate([tr, te])          # <-- THE BUG
    else:
        fit_rows = tr
    mu, _, Vt = fit_basis(X_all[fit_rows], k=k)
    Z_tr = project(X_all[tr], mu, Vt, k)
    Z_te = project(X_all[te], mu, Vt, k)
    cls, cents = nearest_centroid_fit(Z_tr, subject[tr])
    pred = nearest_centroid_predict(Z_te, cls, cents)
    return (pred == subject[te]).mean()


K = 100
rows = []
for name, sp in SPLITS.items():
    rows.append((name, run(sp, K, leak_basis=False), run(sp, K, leak_basis=True)))

print(f'Identity accuracy at k={K}\n')
print(f"{'split':10s} {'honest':>10s} {'leaked basis':>14s} {'inflation':>11s}")
print('-' * 48)
for nm, honest, leaked in rows:
    print(f'{nm:10s} {honest:10.3f} {leaked:14.3f} {leaked - honest:+11.3f}')
print()
print('The random split is also inflated relative to the block split, for a different')
print('reason: neighbouring illumination conditions are near-duplicates.')
print(f"near-duplicate inflation: {rows[0][1] - rows[1][1]:+.3f}")

> **→ In your track.** The `leak_basis=True` branch is three characters of difference and gives
> you a better number. Every project must state, in one sentence, which rows its basis was fitted
> on. In T2, `split='random'` over beats is the near-duplicate leak with a different name — the
> same patient's consecutive beats are as similar as adjacent illumination conditions.

**From here on we use the `block` split only.** It is the honest one.

In [ ]:
SPLIT = SPLITS['block']
tr, te = SPLIT['train'], SPLIT['test']
y_tr, y_te = subject[tr], subject[te]
print(f'train {len(tr)}  test {len(te)}  classes {len(np.unique(y_tr))}')

---
## **F4** · The projection — and *which* components to plot

The class notebook projected onto **modes 5 and 6**, not 1 and 2. That was not arbitrary, and it
is worth understanding before you plot anything.

In Yale B the illumination angle varies far more than the people do, so the leading modes encode
**lighting direction, not identity** — exactly what F3 showed. Plotting PC1–PC2 and captioning it
"class separation" would therefore be the very error this notebook warns about: the dominant
modes are a nuisance variable, and the discriminative signal lives further down the spectrum.

But hard-coding 5 and 6 is only the right answer for *this* dataset and *those* two people. The
defensible move is to **measure discriminability per component and say which you chose and why**.

### The criterion

For each component we compute the ratio of between-class to within-class variance of the training
scores — the one-way ANOVA *F* statistic, `sklearn.feature_selection.f_classif`. A high value
means subjects are far apart relative to how much each subject scatters. Ordering the components
by it answers "where does identity actually live?" with a number rather than an impression.

> **This is model fitting, so it uses training rows only.** Ranking components with the test set
> included is the same leak as §2.4 wearing a different hat — and a subtler one, because feature
> *selection* rarely feels like fitting. It is.

> **→ In your track.** Run this before you plot. If your top-ranked components are 1 and 2, plot
> those and say the criterion agreed. If they are 7 and 12, plot those and say so. In T1 expect
> brightness to dominate the leading modes; in T4 expect total power to. Report the ranking either
> way — a figure showing that the discriminative components are *not* the dominant ones is one of
> the more instructive things you can hand in.

In [ ]:
from sklearn.feature_selection import f_classif

K_VIS = 30
mu_v, _, Vt_v = fit_basis(X_all[tr], k=K_VIS)
Z_tr_v = project(X_all[tr], mu_v, Vt_v, K_VIS)
Z_te_v = project(X_all[te], mu_v, Vt_v, K_VIS)

# discriminability of each component, TRAINING ROWS ONLY
F, _ = f_classif(Z_tr_v, y_tr)
order = np.argsort(F)[::-1]
best = sorted(order[:2])                 # the two most class-discriminative

print('component ranking by between/within-class variance (train only)')
print(f"{'rank':>4}  {'component':>9}  {'F':>10}")
for r, j in enumerate(order[:8]):
    print(f'{r + 1:>4}  {j + 1:>9}  {F[j]:>10.1f}')
print()
print(f'PC1 rank: {int(np.where(order == 0)[0][0]) + 1}   '
      f'PC2 rank: {int(np.where(order == 1)[0][0]) + 1}')
print(f'chosen pair for F4: PC{best[0] + 1} and PC{best[1] + 1}')

In [ ]:
plt.figure(figsize=(7.6, 3.4))
plt.bar(np.arange(1, K_VIS + 1), F, color=['crimson' if (j in best) else '#7f9bc4'
                                           for j in range(K_VIS)])
plt.xlabel('component'); plt.ylabel('F  (between / within class)')
plt.title('Where identity actually lives — discriminability by component')
plt.tight_layout(); plt.show()

In [ ]:
pick = np.unique(y_tr)[:6]
cmap = plt.get_cmap('tab10')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (ia, ib), ttl in zip(
        axes,
        [(0, 1), (best[0], best[1])],
        ['the conventional default', 'chosen by the criterion']):
    for i, s_id in enumerate(pick):
        a, b = y_tr == s_id, y_te == s_id
        ax.scatter(Z_tr_v[a, ia], Z_tr_v[a, ib], s=16, color=cmap(i), alpha=.55,
                   label=f'subj {s_id} (train)')
        ax.scatter(Z_te_v[b, ia], Z_te_v[b, ib], s=52, color=cmap(i), marker='^',
                   edgecolor='k', lw=.5, label=f'subj {s_id} (test)')
    ax.set_xlabel(f'PC {ia + 1}'); ax.set_ylabel(f'PC {ib + 1}')
    ax.set_title(f'PC{ia + 1} vs PC{ib + 1} — {ttl}', fontsize=11)
axes[0].legend(fontsize=6.5, ncol=2, loc='best')
fig.suptitle('F4  six subjects: the default pair vs. the discriminative pair', y=1.02)
fig.tight_layout(); plt.show()

# how separable is each pair, quantified rather than eyeballed?
for lbl, pair in [('PC1-PC2', (0, 1)), (f'PC{best[0]+1}-PC{best[1]+1}', tuple(best))]:
    cls_p, cents_p = nearest_centroid_fit(Z_tr_v[:, list(pair)], y_tr)
    acc = (nearest_centroid_predict(Z_te_v[:, list(pair)], cls_p, cents_p) == y_te).mean()
    print(f'{lbl:>12s}  nearest-centroid accuracy on these 2 components only: {acc:.3f}')

---
## S6–S8 · Predict, beat the baselines, sweep *k*  ·  **F5**

Two predictors (nearest-centroid plus multinomial logistic regression) and three baselines
(majority class, the same classifier on raw pixels with no SVD, and a label-permutation control),
across a range of `k`.

The permutation control is the one people skip. **Any accuracy above the majority rate that
survives shuffled labels is coming from your code, not your data.**

> **A perfect score is a bug report, not a result.** If a classifier reaches 1.000 on held-out
> data, stop and find out why before you celebrate. On the synthetic surrogate logistic regression
> does exactly this, because the surrogate subjects are linearly separable by construction — an
> artefact of how the fake data is made. On the real Yale images it will not. In your track, a
> perfect score almost always means leakage, a duplicated row, or a feature that encodes the
> label.

In [ ]:
K_GRID = [2, 5, 10, 20, 40, 80, 160, 320]
K_GRID = [k for k in K_GRID if k < min(len(tr), n_pixels)]

res = {'nc': [], 'logreg': [], 'perm': []}

for k in K_GRID:
    mu_k, _, Vt_k = fit_basis(X_all[tr], k=k)
    Z_tr = project(X_all[tr], mu_k, Vt_k, k)
    Z_te = project(X_all[te], mu_k, Vt_k, k)

    cls, cents = nearest_centroid_fit(Z_tr, y_tr)
    res['nc'].append((nearest_centroid_predict(Z_te, cls, cents) == y_te).mean())

    lr = LogisticRegression(max_iter=2000)
    lr.fit(Z_tr, y_tr)
    res['logreg'].append(lr.score(Z_te, y_te))

    y_shuf = rng.permutation(y_tr)
    cls_p, cents_p = nearest_centroid_fit(Z_tr, y_shuf)
    res['perm'].append((nearest_centroid_predict(Z_te, cls_p, cents_p) == y_te).mean())

    print(f'k={k:4d}  nc={res["nc"][-1]:.3f}  logreg={res["logreg"][-1]:.3f}'
          f'  permuted={res["perm"][-1]:.3f}')

In [ ]:
# ---------------- baselines ----------------
vals, counts = np.unique(y_tr, return_counts=True)
majority = (y_te == vals[np.argmax(counts)]).mean()

cls_r, cents_r = nearest_centroid_fit(X_all[tr], y_tr)      # raw pixels, NO SVD
raw_acc = (nearest_centroid_predict(X_all[te], cls_r, cents_r) == y_te).mean()

print(f'majority class        : {majority:.3f}')
print(f'nearest-centroid, raw : {raw_acc:.3f}   (no SVD, {n_pixels} features)')
print(f'label permutation     : {max(res["perm"]):.3f}   (worst case over k)')

In [ ]:
# ---------------- F5: performance vs k, with baselines ----------------
plt.figure(figsize=(7, 4.6))
plt.semilogx(K_GRID, res['nc'], 'o-', label='nearest-centroid on SVD coords')
plt.semilogx(K_GRID, res['logreg'], 's-', label='logistic regression on SVD coords')
plt.semilogx(K_GRID, res['perm'], 'x:', color='grey', label='permuted labels (control)')
plt.axhline(majority, color='crimson', ls='--', lw=1.3, label='majority class')
plt.axhline(raw_acc, color='seagreen', ls='-.', lw=1.3, label='same classifier, raw pixels')

best_i = int(np.argmax(res['nc']))
plt.plot(K_GRID[best_i], res['nc'][best_i], '*', ms=18, color='navy', zorder=5)
plt.annotate(f'operating point\nk={K_GRID[best_i]}',
             (K_GRID[best_i], res['nc'][best_i]), textcoords='offset points',
             xytext=(8, -30), fontsize=9)

plt.xlabel('number of components $k$'); plt.ylabel('test accuracy')
plt.title('F5  performance vs. $k$, against three baselines')
plt.legend(fontsize=8, loc='lower right'); plt.ylim(-0.03, 1.03)
plt.tight_layout(); plt.show()

K_STAR = K_GRID[best_i]
print(f'chosen k = {K_STAR}')

**How to justify your operating point.** Not "it was the highest". Say something like: accuracy
plateaus beyond *k* = X, so we take the smallest *k* within one standard error of the best, which
buys a Y-fold reduction in dimension for Z points of accuracy. If accuracy *falls* at large *k*,
say why — later components are noise, and a centroid classifier weights every dimension equally.

> **→ In your track.** If the raw-feature baseline matches your SVD pipeline, report that plainly.
> It is a real finding: the decomposition bought interpretability and speed, not accuracy.

---
## S9 · Interpret honestly  ·  **F6**

In [ ]:
mu_k, _, Vt_k = fit_basis(X_all[tr], k=K_STAR)
Z_tr = project(X_all[tr], mu_k, Vt_k, K_STAR)
Z_te = project(X_all[te], mu_k, Vt_k, K_STAR)
cls, cents = nearest_centroid_fit(Z_tr, y_tr)
pred = nearest_centroid_predict(Z_te, cls, cents)

cm = confusion_matrix(y_te, pred, labels=cls)
per_class = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)

plt.figure(figsize=(6.4, 5.4))
plt.imshow(cm, cmap='Blues')
plt.colorbar(label='count')
plt.xlabel('predicted subject'); plt.ylabel('true subject')
plt.title(f'F6  confusion matrix, k={K_STAR}')
plt.tight_layout(); plt.show()

print(f'overall accuracy  : {(pred == y_te).mean():.3f}')
print(f'balanced accuracy : {balanced_accuracy_score(y_te, pred):.3f}')
print(f'worst class recall: {per_class.min():.3f}  (subject {cls[np.argmin(per_class)]})')
print(f'best  class recall: {per_class.max():.3f}')
print()
print('Report per-class recall, not just overall accuracy. One subject at 0.2 recall is')
print('invisible in an average and would be the whole story in a clinical setting.')

---
## Target B · In-gallery vs. impostor

The second target, and the one that generalizes furthest. Project an image onto the gallery
eigenbasis, reconstruct it, and measure the residual. A face from someone never enrolled lives
partly outside the span of the gallery modes, so it reconstructs worse.

This needs **no labels for the impostors** — it is unsupervised anomaly detection built from the
same decomposition. In imaging it is how you flag a scan that does not belong to your cohort;
in signals it is how you flag a lead that has come loose.

> **→ In your track.** Turk & Pentland call this "distance from face space" [ref 5 in the
> handout]. If your track has a natural out-of-distribution group — a modality your model was not
> trained on, a subject with a different montage — this is a strong stretch result.

In [ ]:
def residual(X, mu, Vt, k):
    Z = (X - mu) @ Vt[:k].T
    recon = Z @ Vt[:k] + mu
    return np.linalg.norm(X - recon, axis=1)


r_gallery = residual(X_all[te], mu_k, Vt_k, K_STAR)       # held-out gallery images
r_impostor = residual(X_all[imp_mask], mu_k, Vt_k, K_STAR)  # never-enrolled subjects

scores = np.concatenate([r_gallery, r_impostor])
labels = np.concatenate([np.zeros(len(r_gallery)), np.ones(len(r_impostor))])
auc = roc_auc_score(labels, scores)

plt.figure(figsize=(6.6, 4.2))
bins = np.linspace(scores.min(), scores.max(), 45)
plt.hist(r_gallery, bins=bins, alpha=.65, label=f'in gallery (n={len(r_gallery)})')
plt.hist(r_impostor, bins=bins, alpha=.65, label=f'impostor (n={len(r_impostor)})')
plt.xlabel(f'reconstruction residual at k={K_STAR}'); plt.ylabel('count')
plt.title(f'Distance from face space — ROC AUC = {auc:.3f}')
plt.legend(); plt.tight_layout(); plt.show()

print(f'ROC AUC          : {auc:.3f}      (0.5 = no signal; AUC has no majority baseline)')
print(f'class balance    : {int((labels==0).sum())} in-gallery vs {int((labels==1).sum())} impostor')
print(f'median residual  : in-gallery {np.median(r_gallery):.1f}  impostor {np.median(r_impostor):.1f}')
print()
print('AUC is used here BECAUSE the classes are so imbalanced that accuracy would be')
print('meaningless — always predicting in-gallery already scores well. This is the same')
print('reason T1 and T4 need per-class rates rather than an overall number.')

---
## Verification — §6 of the handout

Every assertion below must pass before you submit. They are cheap, and they catch the errors that
produce confident, wrong figures.

In [ ]:
ok = []

# 1 — reconstruction and orthonormality
Uc, sc, Vtc = np.linalg.svd(A, full_matrices=False)
ok.append(('reconstruction  ', np.abs(A - (Uc * sc) @ Vtc).max() < 1e-8))
ok.append(('V orthonormal   ', np.allclose(Vtc @ Vtc.T, np.eye(Vtc.shape[0]), atol=1e-8)))
ok.append(('U orthonormal   ', np.allclose(Uc.T @ Uc, np.eye(Uc.shape[1]), atol=1e-8)))

# 2 — Eckart-Young at a specific k
kk = min(50, len(sc) - 1)
err = np.linalg.norm(A - (Uc[:, :kk] * sc[:kk]) @ Vtc[:kk], 'fro')
ok.append(('Eckart-Young    ', abs(err - np.sqrt((sc[kk:] ** 2).sum())) / err < 1e-8))

# 3 — SVD right singular vectors match sklearn PCA (up to sign)
from sklearn.decomposition import PCA
p = PCA(n_components=10, svd_solver='full').fit(X_all[tr])
ok.append(('matches sklearn ', np.allclose(np.abs(p.components_), np.abs(Vtc[:10]), atol=1e-6)))

# 4 — the basis never saw the test rows
ok.append(('no basis leakage', len(np.intersect1d(tr, te)) == 0))

# 5 — permutation control stays at chance
ok.append(('perm at chance  ', max(res['perm']) < majority + 0.10))

# 6 — the model beats every baseline
best = max(max(res['nc']), max(res['logreg']))
ok.append(('beats majority  ', best > majority))
ok.append(('beats permuted  ', best > max(res['perm'])))

print(f'{"check":18s} result')
print('-' * 28)
for name, passed in ok:
    print(f'{name:18s} {"PASS" if passed else "** FAIL **"}')
print()
assert all(p for _, p in ok), 'at least one verification check failed'
print('All verification checks passed.')

---
## The results table your report must contain

In [ ]:
summary = [
    ('majority class',                         majority),
    ('label permutation (worst over k)',        max(res['perm'])),
    ('nearest-centroid, raw pixels (no SVD)',   raw_acc),
    (f'nearest-centroid, {K_STAR} SVD coords',  max(res['nc'])),
    (f'logistic regression, SVD coords',        max(res['logreg'])),
]
print(f'{"model":42s} {"test acc":>9s}')
print('-' * 53)
for nm, v in summary:
    print(f'{nm:42s} {v:9.3f}')
print('-' * 53)
print(f'{"in-gallery vs impostor (ROC AUC)":42s} {auc:9.3f}')
print()
print(f'basis fitted on {len(tr)} training rows only; split = block; seed = {SEED}')
print(f'data source: {"real Yale B" if REAL_DATA else "SYNTHETIC SURROGATE"}')

---
## Transferring this to your track

Every cell above changes in exactly one of three ways. Nothing else moves.

| Stage | Faces | T1 X-ray | T2 ECG beats | T4 sleep EEG | T5 gait | T6 EMG |
|---|---|---|---|---|---|---|
| **Row** | one image | one image | one beat | one 30 s epoch | one gait cycle | one window |
| **Columns** | pixels | pixels | ±250 ms samples | log-power bins | sensors × cycle % | channel features |
| **Target** | identity | pneumonia | beat class | sleep stage | PD vs control | gesture |
| **Split by** | image (identity *is* the target) | provided split | **record** | **subject** | **subject** | **subject** |
| **Alignment** | pre-registered | consistent crop | **R peak** | n/a — use spectra | resample cycles | window + rectify |
| **Scale columns?** | no — same units | no | no | decide: shape vs power | **yes** | **yes** |
| **F3 shows** | mode images | mode images | mode waveforms | mode spectra | mode loading curves | bar per muscle |

### The three things that must survive the transfer

1. **The basis is fitted on training rows only.** Non-negotiable, every track.
2. **Three baselines, always.** Majority, raw-features, permutation.
3. **Per-class rates, not just accuracy.** Every biomedical track here is imbalanced.

### What to do first

Open `BME574_Project1_starter.ipynb`. It already contains the loading, plotting and baseline code
in a much shorter form, with five cells for you to fill in. This notebook is the long version,
kept for reference — do not try to adapt it into a submission.